In [ ]:
""" Run Instructions:
1. Run step by step
2. Select earlier plot in time first
3. Modify variables throughtout depending on desired timeframes and data
4. Script works with "Fitted" and "Ne from Power" files. Does not work with "Resolved Velocity"
"""
import datetime
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import madrigalWeb.madrigalWeb
import os
import h5py
import numpy as np
import pathlib

In [ ]:
#CHANGE ME
user_fullname = "Benjamin Marcotte"
user_email = "bmarc@bu.edu"
user_affiliation = "ISR Summer School 2026"

maddat = madrigalWeb.madrigalWeb.MadrigalData('https://data.amisr.com/madrigal/')
#maddat = madrigalWeb.madrigalWeb.MadrigalData('https://cedar.openmadrigal.org/')

In [ ]:
#instrument codes for AMISR
instcodes={'PFISR':61,
          'RISR-N':91,
          'RISR-C':92}

In [ ]:
# Find first experiment that happened in the following time interval:
st=datetime.datetime(2026, 7, 20, 0,0)
et=datetime.datetime(2026, 7, 22, 23,59)

expList = maddat.getExperiments(instcodes['PFISR'],
                st.year, st.month, st.day, st.hour, st.minute, st.second,
                et.year, et.month, et.day, et.hour, et.minute, et.second)
for exp in expList:
    print(exp)

In [ ]:
# Input Desired Experiment ID from list above into variable below:
selected_experiment_id = 30003610
fileList = maddat.getExperimentFiles(selected_experiment_id)
for file in fileList:
    print(os.path.basename(file.name),'\tkindat:',file.kindat,'desc:',file.kindatdesc)

In [ ]:
# Input desired kindat value from list above into variable below:
# Works with "Fitted" and "Ne From Power" files. Does not work with "Resolved Velocity"
selected_file_kindat = 1000301
acfile=None
for file in fileList:
    if file.kindat == selected_file_kindat:
        acfile=file
        break
        
filename    = acfile.name
outfilename = os.path.basename(acfile.name)
file_path = pathlib.Path(outfilename)

if file_path.is_file():
    print(f"File {outfilename} already downloaded. Skipping download")
else:
    result = maddat.downloadFile(filename,outfilename, user_fullname, user_email, user_affiliation, 'hdf5')
    print(f"Done downloading {outfilename}")

In [ ]:
# Find second experiment that happened in the following time interval:
st=datetime.datetime(2026, 7, 20, 0,0)
et=datetime.datetime(2026, 7, 22, 23,59)

expList = maddat.getExperiments(instcodes['PFISR'],
                st.year, st.month, st.day, st.hour, st.minute, st.second,
                et.year, et.month, et.day, et.hour, et.minute, et.second)
for exp in expList:
    print(exp)

In [ ]:
# Input Desired Experiment ID from list above into variable below:
selected_experiment_id = 30003611
fileList = maddat.getExperimentFiles(selected_experiment_id)
for file in fileList:
    print(os.path.basename(file.name),'\tkindat:',file.kindat,'desc:',file.kindatdesc)

In [ ]:
# Input desired kindat value from list above into variable below:
# Works with "Fitted" and "Ne From Power" files. Does not work with "Resolved Velocity"
selected_file_kindat = 1000301
acfile=None
for file in fileList:
    if file.kindat == selected_file_kindat:
        acfile=file
        break
        
filename    = acfile.name
outfilename2 = os.path.basename(acfile.name)

file_path = pathlib.Path(outfilename2)

if file_path.is_file():
    print(f"File {outfilename2} already downloaded. Skipping download")
else:
    result = maddat.downloadFile(filename,outfilename2, user_fullname, user_email, user_affiliation, 'hdf5')
    print(f"Done downloading {outfilename2}")

In [ ]:
if "nenotr" in outfilename:
    with h5py.File(outfilename,'r') as f:
        PFISR_data = []
        for dat in f['Data/Array Layout'].values():
            outdct={}
            outdct['bid'] = dat['1D Parameters/beamid'][0]
            outdct['azm'] = dat['1D Parameters/azm'][0]
            outdct['elm'] = dat['1D Parameters/elm'][0]
            outdct['ne'] = 10**(dat['2D Parameters/popl'][:])
            outdct['dne'] = 10**(dat['2D Parameters/dpopl'][:])
            
            outdct['range'] = dat['range'][:]
            outdct['altitude'] = outdct['range']*np.sin(np.radians(outdct['elm']))
            tstmp = dat['timestamps'][:]
            outdct['time'] = [datetime.datetime.fromtimestamp(t, tz = datetime.timezone.utc) for t in tstmp]
            PFISR_data.append(outdct) 
elif "fit" in outfilename:
    with h5py.File(outfilename,'r') as f:
        PFISR_data = []
        for dat in f['Data/Array Layout'].values():
            outdct={}
            outdct['bid'] = dat['1D Parameters/beamid'][0]
            outdct['azm'] = dat['1D Parameters/azm'][0]
            outdct['elm'] = dat['1D Parameters/elm'][0]
            outdct['ne'] = dat['2D Parameters/ne'][:]     # different from old SRI madrigal 2
            outdct['dne'] = dat['2D Parameters/dne'][:]   # different from old SRI madrigal 2
            outdct['te'] = dat['2D Parameters/te'][:]
            outdct['dte'] = dat['2D Parameters/dte'][:]
            outdct['ti'] = dat['2D Parameters/ti'][:]
            outdct['dti'] = dat['2D Parameters/dti'][:]
            outdct['vo'] = dat['2D Parameters/vo'][:]
            outdct['dvo'] = dat['2D Parameters/dvo'][:]
            
            outdct['range'] = dat['range'][:]
            outdct['altitude'] = outdct['range']*np.sin(np.radians(outdct['elm']))
            tstmp = dat['timestamps'][:]
            outdct['time'] = [datetime.datetime.fromtimestamp(t, tz = datetime.timezone.utc) for t in tstmp]
            PFISR_data.append(outdct)
else:
    print("Cannot currently handle this file type.")

In [ ]:
if "nenotr" in outfilename2:
    with h5py.File(outfilename2,'r') as f:
        PFISR_data2 = []
        for dat in f['Data/Array Layout'].values():
            outdct={}
            outdct['bid'] = dat['1D Parameters/beamid'][0]
            outdct['azm'] = dat['1D Parameters/azm'][0]
            outdct['elm'] = dat['1D Parameters/elm'][0]
            outdct['ne'] = 10**(dat['2D Parameters/popl'][:])
            outdct['dne'] = 10**(dat['2D Parameters/dpopl'][:])
            
            outdct['range'] = dat['range'][:]
            outdct['altitude'] = outdct['range']*np.sin(np.radians(outdct['elm']))
            tstmp = dat['timestamps'][:]
            outdct['time'] = [datetime.datetime.fromtimestamp(t, tz = datetime.timezone.utc) for t in tstmp]
            PFISR_data2.append(outdct) 
elif "fit" in outfilename:
    with h5py.File(outfilename,'r') as f:
        PFISR_data2 = []
        for dat in f['Data/Array Layout'].values():
            outdct={}
            outdct['bid'] = dat['1D Parameters/beamid'][0]
            outdct['azm'] = dat['1D Parameters/azm'][0]
            outdct['elm'] = dat['1D Parameters/elm'][0]
            outdct['ne'] = dat['2D Parameters/ne'][:]     # different from old SRI madrigal 2
            outdct['dne'] = dat['2D Parameters/dne'][:]   # different from old SRI madrigal 2
            outdct['te'] = dat['2D Parameters/te'][:]
            outdct['dte'] = dat['2D Parameters/dte'][:]
            outdct['ti'] = dat['2D Parameters/ti'][:]
            outdct['dti'] = dat['2D Parameters/dti'][:]
            outdct['vo'] = dat['2D Parameters/vo'][:]
            outdct['dvo'] = dat['2D Parameters/dvo'][:]
            
            outdct['range'] = dat['range'][:]
            outdct['altitude'] = outdct['range']*np.sin(np.radians(outdct['elm']))
            tstmp = dat['timestamps'][:]
            outdct['time'] = [datetime.datetime.fromtimestamp(t, tz = datetime.timezone.utc) for t in tstmp]
            PFISR_data2.append(outdct)
else:
    print("Cannot currently handle this file type.")

In [ ]:
# Input index of desired beam (starting at 0) from list above into variable below
selected_beam_id = 0
beam_data = PFISR_data[selected_beam_id]

In [ ]:
# Input index of desired beam (starting at 0) from list above into variable below for file 2
selected_beam_id = 0
beam_data2 = PFISR_data2[selected_beam_id]

In [ ]:
fig, (ax1, ax2) = plt.subplots(
    1, 2, sharey=True, figsize=(12,5))
fig.subplots_adjust(wspace=0.1)
vmin_value = 0
vmax_value = 2e11
clrs = ax1.pcolormesh(mdates.date2num(beam_data['time']),
                     beam_data['altitude'],
                     beam_data['ne'],
                     vmin=vmin_value,vmax=vmax_value,shading='nearest')
clrs = ax2.pcolormesh(mdates.date2num(beam_data2['time']),
                     beam_data2['altitude'],
                     beam_data2['ne'],
                     vmin=vmin_value,vmax=vmax_value,shading='nearest')


locator = mdates.AutoDateLocator(minticks=3, maxticks=7)
formatter = mdates.ConciseDateFormatter(locator)
ax1.xaxis.set_major_locator(locator)
ax1.xaxis.set_major_formatter(formatter)
ax2.xaxis.set_major_locator(locator)
ax2.xaxis.set_major_formatter(formatter)

ax2.tick_params(left=False, which="both")

ax1.spines["right"].set_visible(False)
ax2.spines["left"].set_visible(False)

ax1.set_xlabel('UT')
ax1.set_ylabel('Altitude (km)')
fig.suptitle('Electron Density vs Range') # Use metadata to make more descriptive

# Add diagonal "break marks" to indicate the gap explicitly. Thanks google gemini
d = 0.5  # Proportion factor for break mark size
kwargs = dict(
    marker=[(-1, -d), (1, d)],
    markersize=12,
    linestyle="none",
    color="k",
    mec="k",
    mew=1,
    clip_on=False,
)

# Draw break marks on the corners of the plots
ax1.plot([1, 1], [0, 1], transform=ax1.transAxes, **kwargs)
ax2.plot([0, 0], [0, 1], transform=ax2.transAxes, **kwargs)

cb=fig.colorbar(clrs)
cb.set_label('Ne (m$^{-3}$)')